# Empirical Asset Pricing via Machine Learning — Replication
**MSDM5054 Statistical Machine Learning, Final Project (Topic 5.1)** — Gu, Kelly & Xiu (2020, *RFS*)

Goal: predict next-month excess returns of individual U.S. stocks from 94 firm characteristics, 74 industry
dummies and 8 macro predictors (920 inputs) with linear models, dimension reduction, trees and neural networks,
evaluated with the paper's **recursive (expanding-window) scheme** on 1987–2016, and study **variable importance**.

| § | Content | Course requirement |
|---|---------|--------------------|
| 0 | Setup | — |
| 1 | Build the panel: characteristics, returns, excess returns, rank-normalisation, Welch–Goyal macro predictors | data, macro predictors |
| 2 | Recursive scheme | recursive performance evaluation |
| 3 | Linear: OLS, OLS-3 (+Huber), ENet+H, PCR, PLS | ≥ 6 methods, OLS-3, robust loss |
| 4 | Trees: RF, GBRT+H | ≥ 6 methods, robust loss |
| 5 | Neural nets NN1–NN5 | NN1–NN5 |
| 6 | Results: $R^2_{oos}$ (all / top-1000 / bottom-1000), Diebold–Mariano, variable importance, complexity | analysis, variable importance |
| 7 | (Optional) decile portfolios | not compulsory |

**Team workflow.** Zhang Qi runs §1 once and shares the code package with `data/panel.parquet` and
`data/macro.csv` through the team OneDrive folder. Everyone unzips it to their own Google Drive
(`My Drive/gkx_replication/`) or uploads it as a Kaggle dataset, sets `MY_MODELS` in §0 and runs only their
own models. Results land in `outputs/<model>/` (one file per test year), which each member uploads to
`results/<name>/` on OneDrive. Every refit is saved immediately, so a disconnect costs at most one refit —
just re-run the cell. Please do not edit `src/`; report bugs in the group chat.

## 0. Setup

In [2]:
import os, sys, json, time, glob, shutil
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
if IN_COLAB:
    # Code package unzipped to My Drive/gkx_replication, with panel.parquet + macro.csv in data/
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/gkx_replication"
elif IN_KAGGLE:
    # Kaggle: upload gkx_replication (code + data/) as a private dataset and add it as input.
    # Code is copied to /kaggle/working so outputs/ is writable; data/ stays read-only.
    src_root = Path(glob.glob("/kaggle/input/**/src/config.py", recursive=True)[0]).parents[1]
    PROJECT_DIR = "/kaggle/working/gkx_replication"
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(src_root, PROJECT_DIR, ignore=shutil.ignore_patterns("data", "outputs*"))
        os.symlink(src_root / "data", Path(PROJECT_DIR) / "data")
else:
    PROJECT_DIR = str(Path.cwd().parent)
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
assert Path("data/panel.parquet").exists() or Path("data/raw").exists(), "put panel.parquet and macro.csv into data/"
# !pip -q install xgboost pyarrow openpyxl

In [8]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dataclasses import replace
from src.config import Config, CHARS, MACRO
from src import data as D, panel as P, recursive as R, metrics as M

cfg = Config()                    # paths relative to PROJECT_DIR, see src/config.py
QUICK = False                     # True: only test years 1987-1988, 1 NN per ensemble
MY_MODELS = ["NN1"]                    # <- each member edits this, e.g. ["OLS", "OLS+H", ...]
if QUICK:
    cfg = replace(cfg, test_end=1988, out_dir="outputs_quick", vi_years=(1987,), vi_sample=50_000)
wg = sorted(Path("data/raw").glob("PredictorData*.xls*"))
if wg:
    cfg = replace(cfg, welch_goyal=str(wg[-1]))
FIG = Path(cfg.out_dir) / "figures"; FIG.mkdir(parents=True, exist_ok=True)
TAB = Path(cfg.out_dir) / "tables"; TAB.mkdir(parents=True, exist_ok=True)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
except Exception as e:
    print("torch not available here (fine for data processing):", type(e).__name__)


GPU: Tesla T4


In [ ]:
print("panel:", Path(cfg.panel_path).exists())
print("macro:", Path(cfg.macro_path).exists())

In [ ]:
from pathlib import Path
print("当前目录：", Path.cwd())
for f in Path("/kaggle/working").rglob("pred_*.parquet"):
    print(f)

## 1. Data (run once by the data owner)
**Inputs to place in `data/raw/`:**
* `datashare.csv` from `datashare_OLD.zip` (94 characteristics, `sic2`, `permno`, `DATE`; 3.7M firm-months, 1957–2016).
* `PredictorData2023.xlsx` (or the latest year) from Amit Goyal's website — monthly Welch–Goyal data.
* Only if `datashare.csv` has no return column: CRSP monthly returns from WRDS (`permno, date, ret, dlret`), set `cfg.crsp_returns`.

**Processing (paper §2.1 and Internet Appendix):**
1. *Timing.* `check_alignment` compares `mom1m` with the return column to decide whether a row's characteristics
   predict the same row's return or the next one — the single most important correctness check of the project.
2. *Target* = return − risk-free rate (`Rfree`, Welch–Goyal).
3. *Characteristics* are ranked cross-sectionally each month into $[-1,1]$; missing values are set to 0 (the median).
4. *Macro predictors* (Welch & Goyal 2008): dp = log(D12) − log(P), ep = log(E12) − log(P), bm, ntis, tbl,
   tms = lty − tbl, dfy = BAA − AAA, svar. They enter with the month in which the forecast is made (no look-ahead).
5. *Features* $z_{i,t} = (1, x_t)\otimes c_{i,t}$ plus industry dummies: 94 × 9 + 74 = 920, built on the fly (`src/panel.py`).

In [ ]:
# Raw data: datashare.csv is downloaded to Colab's local disk (fast; not stored on Drive).
# The Welch-Goyal file (PredictorData20xx.xlsx) must be uploaded to data/raw/ first (only needed by Zhang Qi).
if IN_COLAB and not Path(cfg.panel_path).exists():
    LOCAL = Path("/content/datashare")
    if not list(LOCAL.rglob("*.csv")):
        LOCAL.mkdir(parents=True, exist_ok=True)
        !wget -q -O /content/datashare.zip "https://www.dropbox.com/s/zzgjdubvv23xkfp/datashare.zip?dl=1"
        !unzip -o -q /content/datashare.zip -d /content/datashare
    raw_csv = sorted(LOCAL.rglob("*.csv"), key=lambda f: f.stat().st_size)[-1]
    wg = sorted(Path("data/raw").glob("PredictorData*.xls*"))[-1]
    print("datashare:", raw_csv, f"({raw_csv.stat().st_size/1e9:.2f} GB) | Welch-Goyal:", wg)
    cfg = replace(cfg, raw_csv=str(raw_csv), welch_goyal=str(wg))

In [ ]:
# %pip install "pyarrow==17.0.0"

In [ ]:
import pyarrow
print(pyarrow.__version__)

In [4]:
if not Path(cfg.panel_path).exists():
    t0 = time.time()
    D.build_panel(cfg)              # ~15-30 min; prints the timing (alignment) check
    print(f"built in {(time.time()-t0)/60:.1f} min")
pan = P.Panel(cfg)

panel: 4,339,257 rows | 94 chars | 8 macro | 74 sic2 codes | 920 features | 1927-2020


In [ ]:
# Data checks -> post these numbers in the group chat and use them in report Section 2 / Table 1
pq = pd.read_parquet(cfg.panel_path, columns=["permno", "ret_month", "ret_exc", "size_raw"])
n_month = pq.groupby("ret_month").size()
print(f"rows {len(pq):,} | stocks {pq.permno.nunique():,} | months {len(n_month)} | avg stocks/month {n_month.mean():,.0f}")
print(f"period {pq.ret_month.min():%Y-%m} to {pq.ret_month.max():%Y-%m}")
print(f"excess return: mean {100*pq.ret_exc.mean():.2f}% / month, std {100*pq.ret_exc.std():.2f}%")
q = pq.size_raw.quantile([.01, .5, .99]).round(2).to_dict()
print("mvel1 raw quantiles:", q, "-> LOG scale" if pq.size_raw.max() < 50 else "-> LEVELS")
print("timing check:", json.load(open("data/alignment.json")) if Path("data/alignment.json").exists() else "see build log")
del pq

In [ ]:
# Sample description (report Table 1 / Section 2)
per_month = pd.Series(pan.date).value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(10, 2.8))
ax[0].plot(per_month.index, per_month.values); ax[0].set_title("stocks per month"); ax[0].grid(alpha=.3)
ax[1].hist(np.clip(pan.y, -0.5, 0.5), bins=200); ax[1].set_title("monthly excess return (clipped at ±50%)")
plt.tight_layout(); plt.savefig(FIG / "sample.pdf", bbox_inches="tight")
desc = pd.Series({"firm-months": pan.n, "stocks": len(np.unique(pan.permno)),
                  "avg stocks / month": per_month.mean(), "mean excess ret (%)": 100*pan.y.mean(),
                  "std (%)": 100*pan.y.std(), "skew": pd.Series(pan.y).skew(), "kurtosis": pd.Series(pan.y).kurt()})
display(desc.round(3))
macro = pd.read_csv(cfg.macro_path, index_col=0)
macro.index = pd.PeriodIndex(macro.index, freq="M").to_timestamp()
macro.loc["1957":"2016", MACRO].plot(subplots=True, layout=(2, 4), figsize=(11, 4), legend=False,
                                     title=MACRO, sharex=True)
plt.tight_layout(); plt.savefig(FIG / "macro.pdf", bbox_inches="tight")

## 2. Recursive performance evaluation
Refit every year $Y = 1987,\dots,2016$: training $1957\ldots Y-13$ (expanding), validation the 12 years before $Y$ (rolling),
test year $Y$. Hyper-parameters are chosen on the validation sample only; the test year is never touched before forecasting.

In [5]:
splits = pd.DataFrame([dict(zip(["train", "validation", "test"], cfg.split(Y))) for Y in cfg.refit_years()],
                      index=cfg.refit_years())
splits["n_train"] = [pan.rows(s).stop - pan.rows(s).start for s in splits.train]
display(splits.iloc[[0, 1, 2, -1]])
# display(splits)

,train,validation,test,n_train
1987,"(1957, 1974)","(1975, 1986)","(1987, 1987)",479467
1988,"(1957, 1975)","(1976, 1987)","(1988, 1988)",538166
1989,"(1957, 1976)","(1977, 1988)","(1989, 1989)",597175
2016,"(1957, 2003)","(2004, 2015)","(2016, 2016)",2812624


### Timing test — run this first
One refit per model family on the *largest* training window (2016) gives the per-refit cost; multiply by 30 refits.

In [ ]:
def time_one(name, **kw):
    m = R.make_model(name, cfg, **kw); tr, va, te = (pan.rows(s) for s in cfg.split(2016))
    t0 = time.time(); m.fit(pan, tr, va); m.predict_rows(pan, te); return (time.time() - t0) / 60
TIMING = {"OLS+H": time_one("OLS+H"), "ENet+H": time_one("ENet+H"), "PLS": time_one("PLS")}
# trees / NN (uncomment on a GPU): one grid point each
# TIMING["GBRT+H"] = time_one("GBRT+H", grid={"max_depth": [2], "learning_rate": [0.1]})
# TIMING["NN3 (1 net)"] = time_one("NN3", lrs=(1e-3,), l1s=(1e-4,), n_ensemble=1)
pd.Series(TIMING, name="minutes per refit").round(2)

## 3. Linear models
* **OLS** with all 920 inputs, and **OLS-3** with size (`mvel1`), book-to-market (`bm`) and momentum (`mom12m`), each with
  least squares and with the **Huber loss** (+H, $\xi$ = 99.9% quantile of absolute residuals, IRLS).
* **ENet+H**: elastic net ($\rho=0.5$, $\lambda\in[10^{-4},10^{-1}]$ on standardised inputs) on the Huber-weighted Gram matrix.
* **PCR / PLS**: number of components chosen on the validation sample (PLS = Krylov subspace of $\Sigma_{xx}$ and $\sigma_{xy}$).

All are fitted from streamed $Z^\top WZ$ and $Z^\top Wy$ (921×921), so the 920-column design matrix is never stored.

In [6]:
for name in [m for m in MY_MODELS if m in ("OLS", "OLS+H", "OLS-3", "OLS-3+H", "ENet+H", "PCR", "PLS")]:
    R.run_model(name, pan, cfg)

[OLS] refit 1987: train (1957, 1974) (479,467) | val (1975, 1986) | test (1987, 1987) | tuning
[OLS] 1987: test R2_oos = -181.391%  (0.0 min)
[OLS] 1987: variable importance saved
[OLS] refit 1988: train (1957, 1975) (538,166) | val (1976, 1987) | test (1988, 1988) | tuning
[OLS] 1988: test R2_oos = -95.289%  (0.0 min)
[OLS] refit 1989: train (1957, 1976) (597,175) | val (1977, 1988) | test (1989, 1989) | tuning
[OLS] 1989: test R2_oos = -89.459%  (0.0 min)
[OLS] refit 1990: train (1957, 1977) (656,600) | val (1978, 1989) | test (1990, 1990) | tuning
[OLS] 1990: test R2_oos = -41.335%  (0.0 min)
[OLS] refit 1991: train (1957, 1978) (714,815) | val (1979, 1990) | test (1991, 1991) | tuning
[OLS] 1991: test R2_oos = -13.516%  (0.0 min)
[OLS] refit 1992: train (1957, 1979) (772,497) | val (1980, 1991) | test (1992, 1992) | tuning
[OLS] 1992: test R2_oos = -8.693%  (0.0 min)
[OLS] 1992: variable importance saved
[OLS] refit 1993: train (1957, 1980) (830,637) | val (1981, 1992) | test (1993

In [7]:
from pathlib import Path

for name in MY_MODELS:
    p = Path("outputs") / name
    print(name,
          "pred", len(list(p.glob("pred_*.parquet"))),
          "hparams", len(list(p.glob("hparams_*.json"))),
          "VI", len(list(p.glob("vi_*.csv"))))

OLS pred 30 hparams 30 VI 6
OLS+H pred 30 hparams 30 VI 6
OLS-3 pred 30 hparams 30 VI 6
OLS-3+H pred 30 hparams 30 VI 6
ENet+H pred 30 hparams 30 VI 6
PCR pred 30 hparams 30 VI 6
PLS pred 30 hparams 30 VI 6


## 4. Tree models
* **RF**: 300 trees, depth ∈ {2, 4, 6}, features per split ∈ {10, 30}.
* **GBRT+H**: depth ∈ {1, 2}, learning rate ∈ {0.01, 0.1}, up to 1000 trees with early stopping on validation, pseudo-Huber loss.

Inputs: 94 characteristics + 8 macro + sic2 (trees build interactions themselves; documented deviation).
`tune_every=5` re-tunes every 5 refits to save GPU time (set 1 to follow the paper exactly).

In [ ]:
for name in [m for m in MY_MODELS if m in ("RF", "GBRT+H")]:
    R.run_model(name, pan, cfg, tune_every=5)

## 5. Neural networks NN1–NN5
Hidden layers 32 / 32-16 / 32-16-8 / 32-16-8-4 / 32-16-8-4-2, ReLU, batch norm, L1 penalty ∈ {1e-5, 1e-4, 1e-3},
Adam with learning rate ∈ {0.001, 0.01}, batch 10,000, ≤ 100 epochs, early stopping (patience 5), ensemble of networks
with different seeds (paper: 10). Splitting NN1–NN5 across team members' Colab/Kaggle GPUs is the easiest speed-up.

In [9]:
NN_KW = dict(n_ensemble=1 if QUICK else 5)     # 10 in the paper; 5 halves the cost
for name in [m for m in MY_MODELS if m.startswith("NN")]:
    R.run_model(name, pan, cfg, tune_every=5, **NN_KW)

[NN1] refit 1987: train (1957, 1974) (479,467) | val (1975, 1986) | test (1987, 1987) | tuning
    NN1 lr=0.001 l1=1e-05: val MSE 0.028648 (14 epochs)
    NN1 lr=0.001 l1=0.0001: val MSE 0.028065 (13 epochs)
    NN1 lr=0.001 l1=0.001: val MSE 0.028208 (12 epochs)
    NN1 lr=0.01 l1=1e-05: val MSE 0.027345 (16 epochs)
    NN1 lr=0.01 l1=0.0001: val MSE 0.027032 (12 epochs)
    NN1 lr=0.01 l1=0.001: val MSE 0.026511 (23 epochs)
[NN1] 1987: test R2_oos = 0.287%  (0.8 min)
[NN1] 1987: variable importance saved
[NN1] refit 1988: train (1957, 1975) (538,166) | val (1976, 1987) | test (1988, 1988)
[NN1] 1988: test R2_oos = 0.543%  (0.2 min)
[NN1] refit 1989: train (1957, 1976) (597,175) | val (1977, 1988) | test (1989, 1989)
[NN1] 1989: test R2_oos = 0.002%  (0.2 min)
[NN1] refit 1990: train (1957, 1977) (656,600) | val (1978, 1989) | test (1990, 1990)
[NN1] 1990: test R2_oos = -0.507%  (0.2 min)
[NN1] refit 1991: train (1957, 1978) (714,815) | val (1979, 1990) | test (1991, 1991)
[NN1] 1991:

In [10]:
from pathlib import Path

for name in MY_MODELS:
    p = Path("outputs") / name
    print(name,
          "pred", len(list(p.glob("pred_*.parquet"))),
          "hparams", len(list(p.glob("hparams_*.json"))),
          "VI", len(list(p.glob("vi_*.csv"))))

NN1 pred 30 hparams 30 VI 6


## 6. Results
Run after all model folders are complete (Zhang Qi collects everyone's `outputs/<model>/` folders and shares the
combined `outputs/`; B and C copy it into their own `outputs/` to run the DM and variable-importance cells). $R^2_{oos} = 1-\sum(r-\hat r)^2/\sum r^2$ — benchmark is a **zero** forecast,
not the historical mean (the paper argues the historical mean is so noisy that it would flatter weak models).

In [ ]:
# Progress check: number of finished test years per model (30 = complete)
for m in sorted(p for p in Path(cfg.out_dir).iterdir() if p.is_dir() and p.name not in ("tables", "figures")):
    print(f"{m.name:10s} {len(list(m.glob('pred_*.parquet'))):2d}/30 years | {len(list(m.glob('vi_*.csv')))} VI files")

In [ ]:
r2 = M.r2_table(cfg.out_dir)
display(r2.round(2))
M.to_latex(r2[["All", "Top 1000", "Bottom 1000", "Paper (All)"]], TAB / "tab_r2.tex")
M.plot_r2(r2, FIG / "r2_oos.pdf")
yr = M.r2_by_year(cfg.out_dir); yr.to_csv(TAB / "r2_by_year.csv")
ax = yr.rolling(3, min_periods=1).mean().plot(figsize=(8, 3)); ax.axhline(0, c="k", lw=.6)
ax.set_ylabel("R2_oos (%), 3-yr rolling"); plt.tight_layout(); plt.savefig(FIG / "r2_by_year.pdf", bbox_inches="tight")

In [ ]:
dm = M.dm_matrix(cfg.out_dir)
display(dm.round(2))      # positive: column model beats row model; |DM| > 1.96 significant at 5%
M.to_latex(dm, TAB / "tab_dm.tex")

In [ ]:
ch, mc = M.variable_importance(cfg.out_dir)
display(ch.mean(1).sort_values(ascending=False).head(20).round(3))
M.plot_vi_heatmap(ch, top=20, path=FIG / "vi_heatmap.pdf")
show = [m for m in ["PLS", "ENet+H", "GBRT+H", "NN3"] if m in ch]
if show: M.plot_vi_bars(ch, show, top=20, path=FIG / "vi_bars.pdf")
display(mc.round(3)); M.to_latex(mc, TAB / "tab_vi_macro.tex", fmt="%.3f")
ch.to_csv(TAB / "vi_chars.csv")

In [ ]:
# Characteristic categories (paper's grouping idea): how much importance goes to each family of signals?
GROUPS = {
    "price trend": ["mom1m", "mom6m", "mom12m", "mom36m", "chmom", "indmom", "maxret"],
    "liquidity": ["mvel1", "dolvol", "std_dolvol", "turn", "std_turn", "ill", "zerotrade", "baspread"],
    "risk / volatility": ["retvol", "idiovol", "beta", "betasq"],
    "valuation / fundamentals": ["bm", "ep", "sp", "cfp", "dy", "agr", "roaq", "roeq", "sgr", "lev"],
}
grp = pd.DataFrame({g: ch.loc[[c for c in cs if c in ch.index]].sum() for g, cs in GROUPS.items()})
grp["other"] = 1 - grp.sum(1)
display(grp.round(3)); M.to_latex(grp, TAB / "tab_vi_groups.tex", fmt="%.2f")

In [ ]:
# Model complexity over time (paper Figure 3)
series = {}
for model, key, label in [("ENet+H", "n_nonzero", "ENet+H: # non-zero"), ("PCR", "param", "PCR: # components"),
                          ("PLS", "param", "PLS: # components"), ("GBRT+H", "n_features_used", "GBRT+H: # characteristics"),
                          ("RF", "max_depth", "RF: depth")]:
    s = M.hparam_history(cfg.out_dir, model, key)
    if len(s): series[label] = s
if series: M.plot_complexity(series, FIG / "complexity.pdf")

## 7. (Optional) Decile portfolios — not compulsory for the course

In [ ]:
rows = {}
for m in [x for x in ["OLS-3+H", "PLS", "ENet+H", "RF", "GBRT+H", "NN3"] if x in r2.index]:
    ew, vw = M.decile_portfolios(cfg.out_dir, m, "ew"), M.decile_portfolios(cfg.out_dir, m, "vw")
    rows[m] = {"EW H-L mean (%)": 100*ew["H-L"].mean(), "EW Sharpe": M.sharpe(ew["H-L"]),
               "VW H-L mean (%)": 100*vw["H-L"].mean(), "VW Sharpe": M.sharpe(vw["H-L"])}
port = pd.DataFrame(rows).T; display(port.round(2)); M.to_latex(port, TAB / "tab_portfolios.tex")